# Template matching notebook of selected tilt-series ROIs
This notebook performs template matching on selected regions of interests (ROIs)from a SPED tilt series. The goal is to obtain representative orienation trajectories for the selected crystal regions at each tilt.

This notebook can be run after preprocessing the data. The subsets are then template matched, in order to index the crystals from the different tilts.

Code used in this notebook can be found in "../src".

`src/axis_angle.py`
 - Contains the initial axis-angle PCA candidate-selection method tested
    during method development. This method is kept for reproducibility and
    comparison, but was not used as the final automated selection method.

`src/roi_trajectory_selection.py`
-  Contains the final automated ROI-based candidate trajectory selection used
    for the stereographic analysis.

## Contents
 1. [Load the preprocessed SPED data](#1.-Load-the-data)
 2. [Define/load rois](#2.-Define/load-rois)
 3. [Extract ROI subsets](#3.-Create-subsets)
 4. [Convert the ROI diffraction patterns to polar coordinates](#4.-Polar-transform)
 5. [Create a library of simulated diffraction patterns](#5.-Create-library-of-simulated-diffraction-patterns)
 6. [Load simulations](#7.-Load-simulations)
 7. [Template match the polar-transformed ROIs](#7.-Template-match)
 8. [Select representative oirentation trajectories](#8.-Select-representative-orientation-trajectories)
 9. [Save the results for further analysis](#9.-Save-the-results)

 Two orientation-selection approaches are included in the end of the notebook:
 
 10. [Manual quality-control approach](#10.-Manual-approach)
 11. [Axis-angle candidate-selection method](#1.-Axis-angle-approach)


# Imports

In [ ]:
%matplotlib widget

import json
import pickle
import warnings
from pathlib import Path
import requests
from orix.io import save

warnings.filterwarnings("ignore")

import hyperspy.api as hs
import numpy as np
import matplotlib.pyplot as plt

from diffsims.generators.simulation_generator import SimulationGenerator

from orix.crystal_map import Phase
from orix.quaternion import Orientation, symmetry
from orix.vector import Vector3d
from orix.plot import IPFColorKeyTSL
from orix.sampling import get_sample_reduced_fundamental

### Project imports

In [ ]:
import sys
import importlib

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import src.roi_utils as roi_utils
import src.template_matching as tm
import src.axis_angle as aa
import src.orientation_trajectory_selection as ots

importlib.reload(ots)
importlib.reload(roi_utils)
importlib.reload(tm)
importlib.reload(aa)

from src.roi_utils import (
    load_rois,
    extract_roi_subsets,
)

from src.template_matching import (
    load_polar_rois,
    run_template_matching,
)

from src.axis_angle import (
    run_axis_angle_selection_all_twins,
)

from src.orientation_trajectory_selection import (
    run_roi_trajectory_selection_all_twins,
    print_tilt_step_misorientation,
)

# 1. Load the data

In [ ]:
# load data from 7 tilts
datasets = [
    {"tilt": +10, "path": Path("../../data/20260417_Ingrid/093932/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_18p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": +5,  "path": Path("../../data/20260417_Ingrid/100104/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_13p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": 0,   "path": Path("../../data/20260417_Ingrid/091336/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_8p2_Ty3p8_big_centered_masked.hspy")},
    {"tilt": -5,  "path": Path("../../data/20260417_Ingrid/101949/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_3p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": -10, "path": Path("../../data/20260417_Ingrid/103611/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_m2p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": -15, "path": Path("../../data/20260417_Ingrid/105201/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_m7p2_Ty3p8_cropped_centered_masked.hspy")},
    {"tilt": -20, "path": Path("../../data/20260417_Ingrid/110843/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_m12p2_Ty3p8_cropped_centered_masked.hspy")},
]

results_path = Path("../results")
twin_keys = ["A", "B", "C", "D"]
colors = {
    "A": "red",
    "B": "aqua",
    "C": "mediumorchid",
    "D": "hotpink"
}

In the next cell, the SPED datasets are loaded lazily using HyperSpy. Lazy loading reduces memory usage, which is useful because each dataset contains a scan grid of diffraction patterns.

Each diffraction pattern is normalized by its maximum intensity before template matching: I_norm = I/(I_max + e), where e = 10^(-8) avoids division by zero. This reduces intensity variations between diffraction patterns and imporves robustness of template matching across tilt series.


In [ ]:
lazy = True

signals = {}

for d in datasets:
    
    tilt = d["tilt"]
    datapath = d["path"]

    print(f"Loading tilt {tilt}")

    signal = hs.load(str(datapath), lazy=lazy)

    if lazy:
        signal.rechunk(
            nav_chunks=(32, 32),
            sig_chunks=(32, 32)
        )
    
    # normalize each diffraction pattern
    #signal.map(lambda x: x / (x.max() + 1e-8), inplace=True)

    print(f"  Shape: {signal.data.shape}  dtype: {signal.data.dtype}")
    
    signals[tilt] = signal

# 2. Define/load rois

In [ ]:
# %%
roi_path = results_path / "roi_by_tilt.json"

if not roi_path.exists():
    raise FileNotFoundError(
        f"Could not find ROI file: {roi_path}. "
        "Define and save the ROIs before running this notebook."
    )

roi_by_tilt = load_rois(roi_path)

print(f"Loaded ROIs from: {roi_path}")
print(f"Tilts with ROIs: {sorted(roi_by_tilt.keys())}")

### visualize rois

In [ ]:

"""
for tilt, signal in signals.items():

    print(f"Tilt {tilt}")

    # plot dataset
    signal.plot(norm="symlog")

    rois = roi_by_tilt[tilt]

    # add overlays
    for key, roi in rois.items():
        roi.add_widget(signal, color=colors[key])
"""

# 3. Create subsets

In [ ]:
roi_subsets = extract_roi_subsets(
    signals,
    roi_by_tilt
)

In [ ]:
#plot rois from one tilt to inspect
tilt = 0  # choose your reference tilt

for i, key in enumerate(["A", "B", "C", "D"]):

    subset = roi_subsets[tilt][key]

    subset.plot(norm="symlog", cmap="magma")

plt.tight_layout()

# 4. Polar transform

The ROI diffraction patterns were transformed from Cartesian detector coordinates to polar coordinates before template matching. In polar coordinates, rotational differences between diffraction patterns are represented mainly as shifts along the azimuthal axis, which makes comparison with simulated diffraction patterns more efficient and more robust to in-plane rotations.

In [ ]:
roi_polar = {}

npt = 128
npt_azim=360

polar_root = results_path / "polar_rois"
polar_root.mkdir(parents=True, exist_ok=True)

for tilt, roi_dict in roi_subsets.items():

    print(f"\nProcessing tilt {tilt}")

    roi_polar[tilt] = {}

    tilt_dir = polar_root / f"tilt_{tilt}"
    tilt_dir.mkdir(parents=True, exist_ok=True)

    for key, subset in roi_dict.items():

        print(f"  ROI {key}")

        #prepare data

        subset = subset.copy()
        subset.change_dtype("float32")

        #convert form cartesian to polar coordinates
        subset_pol = subset.get_azimuthal_integral2d(
            npt=npt,
            npt_azim=npt_azim
        )

        # save polar data
        roi_polar[tilt][key] = subset_pol

        subset_pol.save(tilt_dir / f"ROI_{key}_polar.hspy")


# 5. Create library of simlulated diffraction patterns

create simulations

In [ ]:
fname = 'GaAs.cif'
url = "https://materials.springer.com/downloads/track-required/true?path=%2Fassets%2Fsm_isp%2Fcrystallographic%2F311%2Fsm_isp_sd_0311662%2Fsm_isp_sd_0311662_download.cif&componentId=Download+Data+CIF"
r = requests.get(url)
open(fname , 'wb').write(r.content)

GaAs = Phase.from_cif(fname)

In [ ]:
GaAs = Phase.from_cif("../../GaAs.cif")

angular_resolution = 0.5
grid = get_sample_reduced_fundamental(
    resolution=angular_resolution,
    point_group=GaAs.point_group
)

sim_gen = SimulationGenerator(
    accelerating_voltage=200,
    precession_angle=1,
    minimum_intensity=1e-10
)

simulations = sim_gen.calculate_diffraction2d(
    phase=GaAs,
    rotation=grid,
    reciprocal_radius=2,
    with_direct_beam=False
)

orientations = Orientation(
    grid,
    symmetry=GaAs.point_group
)

key_x = IPFColorKeyTSL(GaAs.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(GaAs.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(GaAs.point_group, Vector3d.zvector())

In [ ]:
simulation_root = results_path / "simulation_library"
simulation_root.mkdir(parents=True, exist_ok=True)

sim_path = simulation_root / "GaAs_simulations_angres0p5_r2_mi1em10.pkl"
grid_path = simulation_root / "GaAs_grid_angres0p5.pkl"

In [ ]:

with open(sim_path, "wb") as f:
    pickle.dump(simulations, f)

with open(grid_path, "wb") as f:
    pickle.dump(grid, f)

print("Simulation library saved.")

In [ ]:
#visualize orienations
orientations.scatter('ipf')
orientations

In [ ]:
simulations.plot(interactive=True)

# 5. Load saved simulation
Jump here, if simulations are already defined.

In [ ]:
simulation_root = results_path / "simulation_library"
simulation_root.mkdir(parents=True, exist_ok=True)

sim_path = simulation_root / "GaAs_simulations_angres0p5_r2_mi1em10.pkl"
grid_path = simulation_root / "GaAs_grid_angres0p5.pkl"

In [ ]:
GaAs = Phase.from_cif("../../GaAs.cif")

with open(sim_path, "rb") as f:
    simulations = pickle.load(f)

with open(grid_path, "rb") as f:
    grid = pickle.load(f)

orientations = Orientation(
    grid,
    symmetry=GaAs.point_group,
)

key_x = IPFColorKeyTSL(GaAs.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(GaAs.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(GaAs.point_group, Vector3d.zvector())

print("Simulation library loaded.")

# 6. Template matching

For each ROI and each tilt, the polar-transformed experimental diffraction patterns are compared with the simulated GaAs diffraction library. The best-matching simulated orientations are returned for every scan pixel.

The `n_best = 8` best candidates are kept for each pixel, rather than only the highest-ranked match. This is done because template matching can produce several plausible symmetry-related candidates. Keeping multiple candidates allows the later trajectory-selection step to select the orientation sequence that is most continuous through the tilt series.



# Load polar-transformed ROIs

In [ ]:
polar_root = results_path / "polar_rois"

roi_polar = load_polar_rois(polar_root)

## Compute (template match)

In [ ]:
roi_results = run_template_matching(
    roi_polar,
    simulations,
    n_keep=8,
)

In [ ]:
# quick inspection
"""
tilt = -5
roi_key = "C"

pol = roi_polar[tilt][roi_key]
res = roi_results[tilt][roi_key]

pol.plot()
pol.add_marker(
    res.to_single_phase_polar_markers(
        signal_axes=pol.axes_manager.signal_axes
    )
)

"""

# 7. Automated ROI-based trajectory selection

Template matching returns several possible orientation candidates for each pixel in each ROI. The highest-ranked candidate is not always the most physically consistent through the tilt series, especially when the ROI contains more than one local orientation region or when two templates give similar correlation scores.

To avoid manually choosing a representative pixel, the orientation selection was performed automatically over the ROI. The method first screens pixels using only the highest-ranked candidate. Pixels with low average template-matching score are excluded, and the remaining pixels are ranked by the continuity of their candidate-0 orientation trajectory through the tilt series.

The full `n_best` candidate-sequence optimization is then applied only to the best-ranked pixels. The final representative pixel and candidate sequence are selected as the trajectory with the smallest symmetry-equivalent misorientation discontinuity between neighbouring tilt steps.

In [ ]:
trajectory_results, representative_pixels, trajectory_diagnostics = (
    run_roi_trajectory_selection_all_twins(
        roi_results=roi_results,
        twin_keys=twin_keys,
        symmetry=GaAs.point_group,
        score_percentile=30,
        stride=2,
        keep_n=5,
        rank_weight=0.02,
    )
)

The ROI trajectory selection produced smooth orientation sequences for Twins B, C and D, with neighbouring-step misorientations close to the experimental tilt increment. Twin C is the clearest example of why the automated ROI-based selection was needed. Several shortlisted pixels produced large discontinuities of approximately 50° between neighbouring tilts, indicating a jump to a different orientation branch. The selected pixel, however, gave a continuous trajectory with a maximum neighbouring-step misorientation of approximately 6.3°.

Twin A still shows a larger jump of approximately 18.7°. This trajectory was therefore treated as less ideal and was checked further in the IPF and stereographic validation plots. The selected result was still retained because it gave the lowest trajectory score among the tested ROI pixels.

In [ ]:
# Tilt step validation by calculating the misorientations between the til steps.
# LArge discontinuities may indicate taht selected candidate sequence may have jumped to a different orientation branch.
print_tilt_step_misorientation(
    trajectory_results=trajectory_results,
    twin_keys=twin_keys,
    symmetry=GaAs.point_group,
)

# 9. Save selected orientations
The selected orientation trajectories, representative pixels, and diagnostics are saved for later stereographic analysis. The data are stored in pickle files, as small dictonaries. The crystallographic data is quaternion objects `orix.quaternion.Orientation`, which represent the selected crystal orientation for each tilt. Internally, these orientations are represented by quaternions, while the associated metadata keeps the crystal symmetry. These selected orientations are later used to calculate beam-direction vectors, IPF positions, stereographic trajectories and misorientation angles.

In [ ]:

    
with open(results_path / "orientation_trajectory_results.pkl", "wb") as f:
    pickle.dump(trajectory_results, f)

with open(results_path / "representative_pixels_roi.pkl", "wb") as f:
    pickle.dump(representative_pixels, f)

with open(results_path / "orientation_trajectory_diagnostics.pkl", "wb") as f:
    pickle.dump(trajectory_diagnostics, f)

with open(results_path / "representative_pixels_roi.json", "w") as f:
    json.dump(
        {
            key: [int(px), int(py)]
            for key, (px, py) in representative_pixels.items()
        },
        f,
        indent=4,
    )

print("Saved ROI trajectory selection results.")

## xmaps

In [ ]:
xmap_root = results_path / "roi_xmaps"
xmap_root.mkdir(parents=True, exist_ok=True)

roi_xmaps = {}

for tilt in sorted(roi_results.keys()):

    roi_xmaps[tilt] = {}

    tilt_dir = xmap_root / f"tilt_{tilt}"
    tilt_dir.mkdir(parents=True, exist_ok=True)

    for roi_key in twin_keys:

        print(f"Saving xmap: tilt {tilt}, ROI {roi_key}")

        signal_results = roi_results[tilt][roi_key].deepcopy()

        try:
            signal_results.compute()
        except Exception as error:
            print(error)

        xmap = signal_results.to_crystal_map()

        xmap.prop["index"] = signal_results.isig[0, 0].data.flatten()
        xmap.prop["ci"] = signal_results.isig[1, 0].data.flatten()
        xmap.prop["rotation"] = signal_results.isig[2, 0].data.flatten()
        xmap.prop["mirrored"] = signal_results.isig[3, 0].data.flatten()

        roi_xmaps[tilt][roi_key] = xmap

        save_path = tilt_dir / f"ROI_{roi_key}_xmap.hdf5"

        save(
            save_path,
            xmap,
            overwrite=True,
        )

print("Saved ROI xmaps.")

In [ ]:

with open(results_path / "roi_xmaps.pkl", "wb") as f:
    pickle.dump(roi_xmaps, f)

print("Saved ROI xmaps.")

## 1. IPF-Z plots
sanity check for template matching plotting the highest scoring pixel.

direction = Vector3d.zvector(). which is the beam direction / zone axis direction

plot one twin across all tilts

In [ ]:
twin_key = "C"

tilt_keys = sorted(roi_results.keys())  # assumes roi_results[tilt]
n = len(tilt_keys)

fig = plt.figure(figsize=(2*n, 4))

for i, tilt in enumerate(tilt_keys):

    res = roi_results[tilt][twin_key]

    # ori shape: nav_x, nav_y, n_best

    oris = res.to_single_phase_orientations()[:, 0]
    ori = res.to_single_phase_orientations()[7, 0, 0]
    

    ax = fig.add_subplot(1, n, i + 1, projection="ipf", symmetry=GaAs.point_group)

    ax.scatter(
        ori,
        cmap="inferno",
    )

    ax.set_title(f"\nTilt {tilt} | Twin {twin_key}\n")

plt.tight_layout()
plt.show()

In [ ]:
# Plot one twin across all tilts
twin_key = "C"
tilt_keys = np.array(sorted(roi_results.keys()))

all_oris = []
all_tilts = []

for tilt in tilt_keys:
    res = roi_results[tilt][twin_key]
    ori = res.to_single_phase_orientations()[6, 7, 0]

    all_oris.append(ori)
    all_tilts.append(tilt)

oris_all = Orientation(
    np.stack([o.data.squeeze() for o in all_oris]),
    symmetry=GaAs.point_group
)

tilt_colors = np.array(all_tilts)

cmap = plt.cm.plasma
norm = plt.Normalize(vmin=tilt_keys.min(), vmax=tilt_keys.max())
rgba = cmap(norm(tilt_colors))

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="ipf", symmetry=GaAs.point_group)

ax.scatter(
    oris_all,
    color=rgba,
    alpha=0.9,
    s=80
)

ax.set_title(f"Twin {twin_key}\n")

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, shrink=0.9)
cbar.set_label("Tilt")
cbar.set_ticks(tilt_keys)
cbar.set_ticklabels([str(t) for t in tilt_keys])

plt.tight_layout()
plt.show()

In [ ]:
# Plot one twin across all tilts
twin_key = "C"
tilt_keys = np.array(sorted(roi_results.keys()))

# data
all_oris = []
all_tilts = []

for tilt in tilt_keys:

    res = roi_results[tilt][twin_key]

    oris = res.to_single_phase_orientations()[0, 0, 0]

    all_oris.append(oris)

    n = oris.data.shape[0]
    all_tilts.append(np.full(n, tilt))

oris_all = Orientation(
    np.concatenate([o.data for o in all_oris]),
    symmetry=symmetry.Oh
)

tilt_colors = np.concatenate(all_tilts)

# colormap
cmap = plt.cm.plasma
norm = plt.Normalize(
    vmin=tilt_keys.min(),
    vmax=tilt_keys.max()
)

rgba = cmap(norm(tilt_colors))

# ---- plot ----
fig = plt.figure(figsize=(5, 4))

ax = fig.add_subplot(
    111,
    projection="ipf",
    symmetry=symmetry.Oh
)

ax.scatter(
    oris_all,
    color=rgba,
    alpha=0.9,
    s=80
)

ax.set_title(f"Twin {twin_key}")

# colorbar
sm = plt.cm.ScalarMappable(
    norm=norm,
    cmap=cmap
)

sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    shrink=0.8
)

cbar.set_label("Tilt")
cbar.set_ticks(tilt_keys)
cbar.set_ticklabels([str(t) for t in tilt_keys])

plt.tight_layout()
plt.show()

m-3m

Search the roi subsets for the pixel and candidate sequence with the smoothest orientation trajectory.

# Template match test for one



In [ ]:
tilt = -15
key = "C"

n_keep = 8
npt = 128
npt_azim = 360

subset = roi_subsets[tilt][key].copy()

subset.change_dtype("float32")

subset_pol = subset.get_azimuthal_integral2d(
    npt=npt,
    npt_azim=npt_azim
)

res = subset_pol.get_orientation(
    simulation=simulations,
    n_best=8,
    frac_keep=1.0
)

subset_pol.plot(norm="symlog")

subset_pol.add_marker(
    res.to_single_phase_polar_markers(
        signal_axes=subset_pol.axes_manager.signal_axes
    )
)

subset = roi_subsets[-15]["C"]
subset.plot(cmap='viridis_r', norm='symlog')
subset.add_marker(res.to_markers(annotate=True))

roi_results = {}
roi_results[tilt] = {}
res = subset_pol.get_orientation(
            simulation=simulations,
            n_best=n_keep,
            frac_keep=1.0
        )

        roi_results[tilt][key] = res

check one match 

In [ ]:
pol_0_A = roi_polar[-15]["C"]


pol_0_A.plot()

pol_0_A.add_marker(roi_results[-15]["C"].to_single_phase_polar_markers(signal_axes=pol_0_A.axes_manager.signal_axes))
res_0_A= pol_0_A.get_orientation(simulations, n_best=n_keep, frac_keep=1.0)
res_0_A
res_0_A = roi_subsets[-15]["C"]

res_0_A.plot(cmap='viridis_r', norm='log')
res_0_A.add_marker(roi_results[-15]["C"].to_markers(annotate=True))




In [ ]:
from orix.io import load

In [ ]:
# %%
# -------------------------
# Load saved ROI xmaps
# -------------------------

xmap_root = results_path / "roi_xmaps"

roi_xmaps = {}

for tilt in sorted(roi_results.keys()):

    roi_xmaps[tilt] = {}

    for roi_key in twin_keys:

        path = xmap_root / f"tilt_{tilt}" / f"ROI_{roi_key}_xmap.hdf5"

        roi_xmaps[tilt][roi_key] = load(path)

print("Loaded ROI xmaps.")

In [ ]:
# %%
tilt = -5
roi_key = "C"

xmap = roi_xmaps[tilt][roi_key]

rgb = key_z.orientation2color(
    xmap.orientations,
)

rgb_map = rgb.reshape(
    xmap.shape + (3,),
)

plt.figure(figsize=(5, 4))
plt.imshow(rgb_map)
plt.title(f"ROI {roi_key}, tilt {tilt}: IPF-Z")
plt.axis("off")
plt.show()

In [ ]:
# %%
plt.figure(figsize=(5, 4))
plt.imshow(
    xmap.prop["ci"].reshape(xmap.shape),
    cmap="magma",
)
plt.colorbar(label="CI")
plt.title(f"ROI {roi_key}, tilt {tilt}: confidence")
plt.axis("off")
plt.show()

In [ ]:

# Plot IPF-Z xmaps for all tilts and ROIs
# -------------------------

tilts = sorted(roi_xmaps.keys())

fig, axes = plt.subplots(
    len(twin_keys),
    len(tilts),
    figsize=(2.2 * len(tilts), 2.2 * len(twin_keys)),
    squeeze=False,
)

for row, roi_key in enumerate(twin_keys):

    for col, tilt in enumerate(tilts):

        ax = axes[row, col]

        xmap = roi_xmaps[tilt][roi_key]

        rgb = key_z.orientation2color(
            xmap.orientations,
        )

        rgb_map = rgb.reshape(
            xmap.shape + (3,),
        )

        ax.imshow(rgb_map)

        if row == 0:
            ax.set_title(f"Tilt {tilt}")

        if col == 0:
            ax.set_ylabel(f"ROI {roi_key}")

        ax.set_xticks([])
        ax.set_yticks([])

        # Mark automated representative pixel if available
        if "representative_pixels" in globals():
            if roi_key in representative_pixels:
                px, py = representative_pixels[roi_key]
                if px < xmap.shape[0] and py < xmap.shape[1]:
                    ax.scatter(
                        py,
                        px,
                        marker="x",
                        s=70,
                        c="black",
                    )

plt.suptitle("IPF-Z maps from template matching")
plt.tight_layout()

plt.show()

In [ ]:
representative_pixels = {
    "A": (0, 0),
    "B": (6, 2),
    "C": (6, 6),
    "D": (2, 2)
}